In [1]:
import numpy as np

x = np.array([2, 4, 6])
y = np.array([80, 140, 190])
n = len(x)

lr = 0.01
w = 25
b = 30


y_hat = w * x + b

error = y - y_hat
mse = np.mean(error ** 2)

assert abs(mse - 66.667) < 0.001

dw = - 2 * np.mean(x * error)
db = - 2 * np.mean(error)

w -= lr * dw
b -= lr * db

assert abs(w - 25.667) < 0.001
assert abs(b - 30.133) < 0.001

In [2]:
import numpy as np

x = np.array([2, 4, 6])
y = np.array([80, 140, 190])
n = len(x)

lr = 0.06
w = 25
b = 30

mses = []

for _ in range(1000):
    y_hat = w * x + b
    error = y - y_hat
    mse = np.mean(error ** 2)
    mses.append(mse)
    dw = - 2 * np.mean(x * error)
    db = - 2 * np.mean(error)
    w -= lr * dw
    b -= lr * db

print(f"w: {w}")
print(f"b: {b}")
print(f"mse: {mse}")

w: -3.203407790959749e+128
b: -6.9150266593166955e+127
mse: 1.161875211855281e+258


In [3]:
import pandas as pd

train = pd.read_csv("../data/train_features.csv")
val = pd.read_csv("../data/val_features.csv")

y_train = train["price"]
X_train = train.drop(columns = ["price", "id"]).astype(float)

y_val = val["price"]
X_val = val.drop(columns = ["price", "id"]).astype(float)

assert len(X_train.columns) == 132
assert all(col not in X_train.columns for col in ["price", "id"])

print(X_train.max().sort_values(ascending = False).head(10))

maximum_nights          1489.00000
host_months              185.00000
amenities_count           96.00000
bedrooms                  50.00000
beds                      50.00000
latitude                  40.53553
bath_num                  38.00000
minimum_nights            32.00000
accommodates              16.00000
review_scores_rating       5.00000
dtype: float64


In [4]:
X_train_std = (X_train - X_train.mean()) / X_train.std()
X_val_std = (X_val - X_train.mean()) / X_train.std()
assert X_train_std.mean().abs().max() < 1e-9
assert abs(X_train_std.std() - 1).max() < 1e-9
print(X_val_std.mean().abs().sort_values(ascending = False).head(5))

has_reviews       0.057534
nb_San Isidro     0.056836
nb_Arapiles       0.054129
nb_Aravaca        0.052355
pt_Entire home    0.050787
dtype: float64


In [5]:
X = X_train_std.to_numpy()
y = y_train.to_numpy()
w = np.zeros(X.shape[1])
b = 0
lr = 0.1
n = len(X)

mses = []

for _ in range(10000):
    y_hat = X @ w + b
    error = y - y_hat
    mse = np.mean(error ** 2)
    mses.append(mse)
    dw = - 2 / n * X.T @ error
    db = - 2 * np.mean(error)
    w -= lr * dw
    b -= lr * db

print(f"mse: {mse}")
print(f"rmse: {mse ** 0.5}")

mse: 13774.149378412176
rmse: 117.36332211731302


In [6]:
print(y.std())

148.14030779104016


In [7]:
X_va = X_val_std.to_numpy()
y_va = y_val.to_numpy()
mse = np.mean((y_va - (X_va @ w + b)) ** 2)
rmse = mse ** 0.5
R2 = 1 - mse / (y_va.std() ** 2)

print(f"rmse: {rmse}")
print(f"R2: {R2}")

rmse: 121.27630941382458
R2: 0.3584902597607772


In [8]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)

assert np.abs(w - model.coef_).max() < 1e-4
assert abs(b - model.intercept_) < 1e-8
assert abs(np.mean((y_va - model.predict(X_va)) ** 2) ** 0.5 - rmse) < 0.01

In [9]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x_toy = np.array([2, 4, 6])
w_toy = 1
b_toy = -4
z = w_toy * x_toy + b_toy
sig = sigmoid(z)

assert abs(sig[1] - 0.5) < 1e-9
assert abs(sig[2] - 0.881) < 1e-3

In [10]:
y_toy = np.array([0, 1, 1])
loss = -(y_toy * np.log(sig) + (1 - y_toy) * np.log(1 - sig)).mean()
assert abs(loss - 0.316) < 1e-3

In [11]:
grad_w = np.mean((sig - y_toy) * x_toy)
grad_b = np.mean(sig - y_toy)
assert abs(grad_w + 0.825) < 1e-3
assert abs(grad_b + 0.167) < 1e-3
lr = 0.1
new_w_toy = w_toy - lr * grad_w
new_b_toy = b_toy - lr * grad_b

z = new_w_toy * x_toy + new_b_toy
new_sig = sigmoid(z)
new_loss = -(y_toy * np.log(new_sig) + (1 - y_toy) * np.log(1 - new_sig)).mean()
assert new_loss < loss

In [12]:
threshold = train["price"].median()
print(threshold)

y_cls = (train["price"] > threshold).astype(int).to_numpy()
y_va_cls = (val["price"] > threshold).astype(int).to_numpy()

assert np.isin(y_cls, [0, 1]).all()
assert abs((y_cls == 1).mean() - 0.5) < 0.05

print(threshold)
print((y_cls == 1).mean())
print((y_va_cls == 1).mean())

125.75
125.75
0.49964697575900213
0.5073206442166911


In [13]:
w_log = np.zeros(X.shape[1])
b_log = 0
n = len(X)
lr = 0.1

for i in range(10000):
    p = sigmoid(X @ w_log + b_log)
    grad_w = X.T @ (p - y_cls) / n
    grad_b = np.mean(p - y_cls)
    w_log -= lr * grad_w
    b_log -= lr * grad_b

    if i == 0:
        init_loss = -(y_cls * np.log(p) + (1 - y_cls) * np.log(1 - p)).mean()
        assert abs(init_loss - 0.693) < 1e-3

    if (i + 1) % 1000 == 0:
        loss = -(y_cls * np.log(p) + (1 - y_cls) * np.log(1 - p)).mean()
        print(loss)

assert loss < init_loss

0.4231203037242669
0.42303649886132505
0.4230026048801684
0.4229806628195934
0.4229650049385057
0.4229534036998585
0.42294464226993944
0.4229379471313554
0.4229327881851067
0.42292878665508254


In [14]:
base_acc = (y_va_cls == 1).mean() * 100

p_va = sigmoid(X_va @ w_log + b_log)
pred_va = (p_va > 0.5).astype(int)
acc = (pred_va == y_va_cls).mean() * 100
loss = -(y_va_cls * np.log(p_va) + (1 - y_va_cls) * np.log(1 - p_va)).mean()
assert acc > base_acc

print(acc)
print(loss)

78.7701317715959
0.4298591410860645


In [15]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(C=np.inf)
log_model.fit(X, y_cls)

print(abs(log_model.coef_[0] - w_log).max())
print(abs(log_model.intercept_[0] - b_log))
model_p = log_model.predict_proba(X_va)[:, 1]
model_pred = (model_p > 0.5).astype(int)
print((model_pred == y_va_cls).mean() * 100)
print(-(y_va_cls * np.log(model_p) + (1 - y_va_cls) * np.log(1 - model_p)).mean())

0.20730704298116837
0.0172640252958407
78.7701317715959
0.42988368443177977


In [16]:
col = X_train.columns[np.argmax(abs(log_model.coef_[0] - w_log))]
print(col)
print((X_train[col] == 1).mean())
l = y_cls[X_train[col] == 1]
print((l == 1).mean())

pt_Shared room in bed and breakfast
0.003844041735310269
0.0


In [17]:
keep = ~X_train.columns.isin(["pt_Shared room in bed and breakfast", "pt_Private room in guest suite"])
assert keep.sum() == X.shape[1] - 2

assert abs(model_p - p_va).max() < 0.05
assert abs(log_model.coef_[0][keep] - w_log[keep]).max() < 0.1
assert abs(log_model.intercept_[0] - b_log) < 0.05
assert abs((model_pred == y_va_cls).mean() * 100 - acc).max() < 0.5
assert abs(-(y_va_cls * np.log(model_p) + (1 - y_va_cls) * np.log(1 - model_p)).mean() - loss).max() < 1e-3